In [ ]:
import os
os.environ["OPENAI_API_KEY"]="open ai api key"
os.environ["LANGCHAIN_API_KEY"]="langchain api key"

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

faiss_index = FAISS.load_local(
    "faiss_store",
    embedding_model,
    allow_dangerous_deserialization=True
)

f:\Projects\Langgraph\lang\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

embedding_model1= OpenAIEmbeddings(model="text-embedding-3-large")

faiss_index1= FAISS.load_local(
    "faiss_store1",
    embedding_model1,
    allow_dangerous_deserialization=True
)

In [4]:
all_docs = list(faiss_index1.docstore._dict.values())

In [5]:
from langchain_core.documents import Document

documents = [
    Document(page_content=doc.page_content, metadata={"source": "pdf", "chunk_id": doc.metadata["chunk_id"]})
    for doc in all_docs
]

In [6]:
from langchain_community.retrievers import BM25Retriever
retriever = BM25Retriever.from_documents(documents,k=7)

In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_fn = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vectordb2 = FAISS.load_local("mpnet", embedding_fn, allow_dangerous_deserialization=True)

C:\Users\DELL\AppData\Local\Temp\ipykernel_20928\4293479046.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_fn = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [8]:
# fused_scores={}
# for res in (result1,result2,result3):
#     for rank, doc in enumerate(res):
#         id=doc.metadata["chunk_id"]
#         if id  not in fused_scores:
#             fused_scores[id]=0
#         fused_scores[id]+=1/(rank+60)

# reranked_results=[(doc,score) for doc,score in sorted(fused_scores.items(), key=lambda x:x[1],reverse=True)]
# r=next((d for d in result1+result2+result3 if d.metadata["chunk_id"] == reranked_results[0][0]),None)



In [9]:
import pandas as pd
df=pd.read_excel("Question and answer.xlsx")

In [10]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo")

In [11]:
con=[]
ans=[]
for i,query in enumerate(list(df["Question "])):
    print(f"question {i} = {query}")
    results1 = faiss_index.similarity_search(query, k=7)
    results2 = faiss_index1.similarity_search(query, k=7)
    results3 = retriever.invoke(query)
    results4 = vectordb2.similarity_search(query, k=7)
    fused_scores={}
    for res in (results1,results2,results3,results4):
        for rank, doc in enumerate(res):
            id=doc.metadata["chunk_id"]
            if id  not in fused_scores:
                fused_scores[id]=0
            fused_scores[id]+=1/(rank+60)

    reranked_results=[(doc,score) for doc,score in sorted(fused_scores.items(), key=lambda x:x[1],reverse=True)]
    r=next((d for d in results1+results2+results3+results4 if d.metadata["chunk_id"] == reranked_results[0][0]),None)
    context=r.page_content
    con.append(context)
    print(f"context {i} = {context}")
    prompt = f"""
    Use the context to answer the question, answer lies only in the context.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    answer=llm.invoke(prompt).content
    print(f"answer {i} = {answer}")
    ans.append(answer)


question 0 = What does Article 150 of the Constitution of India provide regarding Government Accounts?
context 0 = 1.1.1   Article 150 of the Constitution of India provides for the maintenance of Government 
Accounts in such form as the President may, on the advice of the Comptroller and Auditor -
General of India, prescribe.   In exercise of these powers , basic rules relating to the Form of 
Accounts were  framed  in the form of ‘ Government Accounting Rules ’ (GAR) . The Civil Accounts 
Manual is intended to guide the Civil Ministries/ Departments of Central Government in carrying
answer 0 = Article 150 of the Constitution of India provides for the maintenance of Government Accounts in such form as the President may, on the advice of the Comptroller and Auditor-General of India, prescribe.
question 1 = Where does the payment process in PFMS start?
context 1 = 2.2 PROCESSING OF CLAIMS IN PAOs THROUGH PFMS  
Introduction  
2.2.1  The payment process in PFMS starts at Program Division 

In [12]:
df["context"]=con

In [13]:
df["answer"]=ans

In [14]:
df.to_excel("our_RAG71.xlsx")